# Setup

In [1]:
%run notebook_setup.py

import pandas as pd
import numpy as np
from scipy.stats import trim_mean
import polars as pl
import pandas as pd
import logging
import gc
import random
import datetime as dt

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

from src.config.dir_config import OUTPUT_PATH_DEMAND_SUMMARY
from src.config.bigquery_config import CREDENTIALS_GBQ, PROJECT_ID_GBQ
from src.utils.read_data import read_data
from src.utils.setup_logging import setup_logging
from src.utils.create_week_date import add_week_start_date
from src.utils.export_as_excel import export_dataframes_as_tables


import logging

setup_logging()

Now you can import modules from the project root: /bi/workspace/Projects/Forecast/forecast


In [2]:
def calc_wape(total_error_abs, total_sales):
    return round(total_error_abs / total_sales, 4) if total_sales != 0 else np.nan

def calc_bias(total_error, total_sales):
    return round(total_error / total_sales, 4) if total_sales != 0 else np.nan

def calc_mae(error_abs):
    if len(error_abs) == 0:
        return np.nan
    return round(np.mean(error_abs), 4)

def calc_rmse(error):
    if len(error) == 0:
        return np.nan
    return round(np.sqrt(np.mean(np.square(error))), 4)

In [3]:
def eval_forecast(df, forecast_dict):
    """
    Calcula WAPE, BIAS, MAE y RMSE para cada forecast usando
    las funciones definidas y validando con distintas columnas de ventas reales.
    
    Args:
        df (pd.DataFrame): dataframe con ventas reales y forecasts
        forecast_dict (dict): diccionario {forecast_col: real_col}
    
    Returns:
        pd.DataFrame con métricas por método
    """
    results = []

    for forecast_col, real_col in forecast_dict.items():
        actual = df[real_col].fillna(0).astype(float)
        forecast = df[forecast_col].fillna(0).astype(float)

        error = forecast - actual
        error_abs = np.abs(error)

        total_error_abs = error_abs.sum()
        total_error = error.sum()
        total_sales = actual.sum()

        results.append({
            "forecast": forecast_col,
            "tipo_venta_real": real_col,
            "wape": calc_wape(total_error_abs, total_sales),
            "bias": calc_bias(total_error, total_sales),
            "mae": calc_mae(error_abs),
            "rmse": calc_rmse(error)
        })
    
    return pd.DataFrame(results)

In [4]:
import numpy as np
import pandas as pd
from numba import jit
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

@jit(nopython=True)
def ses(serie, alpha):
   n = len(serie)
   fitted = np.zeros(n)
   if n > 0:
       fitted[0] = serie[0]
       for i in range(1, n):
           fitted[i] = alpha * serie[i-1] + (1 - alpha) * fitted[i-1]
   return fitted

@jit(nopython=True)
def holt(serie, alpha, beta):
    n = len(serie)
    fitted = np.zeros(n)
    if n < 2:
        fitted[0] = serie[0] if n == 1 else 0.0
        return fitted
    
    # Inicialización más robusta
    level = serie[0]
    # Si hay solo 2 puntos o tendencia inicial es negativa, usar tendencia pequeña
    if n == 2 or serie[1] <= serie[0]:
        trend = max(serie[1] - serie[0], 0.01)  # Mínimo 0.01 para evitar 0
    else:
        trend = serie[1] - serie[0]
    
    fitted[0] = level
    if n > 1:
        fitted[1] = level + trend
    
    for i in range(2, n):
        prev_level = level
        level = alpha * serie[i-1] + (1 - alpha) * (level + trend)
        trend = beta * (level - prev_level) + (1 - beta) * trend
        fitted[i] = max(level + trend, 0.0)  # Evitar valores negativos
    
    return fitted

@jit(nopython=True)
def croston(serie, alpha):
   n = len(serie)
   fitted = np.zeros(n)
   a, q = 0.0, 1.0
   initialized = False
   
   for i in range(n):
       if serie[i] > 0:
           if not initialized:
               a, q = serie[i], 1.0
               initialized = True
           else:
               a = a + alpha * (serie[i] - a)
               q = q + alpha * (1 - q)
       
       if initialized:
           fitted[i] = a / q
       else:
           fitted[i] = 0.0
   return fitted

@jit(nopython=True)
def tsb(serie, alpha_d, alpha_p):
   n = len(serie)
   fitted = np.zeros(n)
   z, p = 0.0, 0.0
   first = True
   
   for i in range(n):
       demanda = 1.0 if serie[i] > 0 else 0.0
       if first:
           z, p = serie[i], demanda
           first = False
       else:
           z = z + alpha_d * (serie[i] - z)
           p = p + alpha_p * (demanda - p)
       fitted[i] = z * p
   return fitted

def procesar_grupo(args):
   grupo_data, value_col, alpha, beta, gamma, alpha_d, alpha_p = args
   serie = grupo_data[value_col].values.astype(np.float64)
   
   # Aplicar métodos
   ses_fit = ses(serie, alpha)
   holt_fit = holt(serie, alpha, beta)
   croston_fit = croston(serie, alpha)
   tsb_fit = tsb(serie, alpha_d, alpha_p)
   
   # Crear resultado
   resultado = grupo_data.copy()
   resultado["SES_mean"] = np.round(np.maximum(ses_fit, 0), 3)
   resultado["Holt_mean"] = np.round(np.maximum(holt_fit, 0), 3)
   resultado["Croston_mean"] = np.round(np.maximum(croston_fit, 0), 3)
   resultado["TSB_mean"] = np.round(np.maximum(tsb_fit, 0), 3)
   
   return resultado

def aplicar_promedios(df, id_cols, week_col, value_col, alpha=0.2, beta=0.1, gamma=0.1, season_length=4, alpha_d=0.1, alpha_p=0.1):
   # Preparar argumentos para procesamiento paralelo
   grupos = [(grupo, value_col, alpha, beta, gamma, alpha_d, alpha_p) 
             for _, grupo in df.groupby(id_cols, observed=True)]
   
   # Procesamiento paralelo
   n_cores = min(mp.cpu_count(), len(grupos))
   with ProcessPoolExecutor(max_workers=n_cores) as executor:
       resultados = list(executor.map(procesar_grupo, grupos))
   
   # Concatenar resultados
   df_resultado = pd.concat(resultados, ignore_index=True)
   
   # Aplicar regla de semanas <= 2
   mask_weeks = df_resultado[week_col] <= 2
   df_resultado.loc[mask_weeks, ["SES_mean", "Holt_mean", "Croston_mean", "TSB_mean"]] = 0
   
   return df_resultado

# Load data

In [5]:
data_all = pd.read_parquet('../sandbox/data_genex_venta_transfer.parquet')

In [6]:
df_sample = data_all.query(
    "cod_sucursal == 14 and cod_producto == 680279 and cod_talla == 105"
)[['nombre_sucursal','cod_producto','nom_talla','week_number','weekly_sales']].reset_index(drop=True)

df_result = aplicar_promedios(
    df_sample,
    id_cols=["nombre_sucursal","cod_producto","nom_talla"],
    week_col="week_number",
    value_col="weekly_sales",
    alpha=0.4,
    beta=0.2,
    alpha_d=0.1,
    alpha_p=0.1,
    season_length=4
)

df_result

,nombre_sucursal,cod_producto,nom_talla,week_number,weekly_sales,SES_mean,Holt_mean,Croston_mean,TSB_mean
0,OSORNO,680279,L,1,3,0.000,0.000,0.000,0.000
1,OSORNO,680279,L,2,1,0.000,0.000,0.000,0.000
2,OSORNO,680279,L,3,0,2.200,2.055,2.200,2.268
3,OSORNO,680279,L,4,0,1.320,0.918,2.200,1.837
4,OSORNO,680279,L,5,0,0.792,0.162,2.200,1.488
5,OSORNO,680279,L,6,0,0.475,0.000,2.200,1.205
6,OSORNO,680279,L,7,0,0.285,0.000,2.200,0.976
7,OSORNO,680279,L,8,2,0.171,0.000,2.120,1.066
8,OSORNO,680279,L,9,0,0.903,0.280,2.120,0.863
9,OSORNO,680279,L,10,0,0.542,0.027,2.120,0.699


In [7]:
factores_historicos = pd.read_parquet('../data/processed/factores_invierno_2024.parquet')

data_all = pd.merge(
    data_all,
    factores_historicos,
    on=['cod_ano_comercial', 'cod_semana','nombre_depto','nombre_linea'],
    how='left'
)

# Calculos

In [ ]:
data_all = aplicar_promedios(
    data_all,
    id_cols=["nombre_sucursal","cod_producto","nom_talla"],
    week_col="week_number",
    value_col="weekly_sales",
    alpha=0.4,
    beta=0.2,
    alpha_d=0.1,
    alpha_p=0.1
)


# Modelacion forecasts

In [ ]:
cond1 = data_all["can_final"] > 0 # Que saliera repo
cond2 = data_all["repo_x_dda"] > 0 # Que hubiera repo x dda calculada
cond3 = data_all["repo_x_dda"] >= data_all["repo_x_ume"] # Que la repo sea por demanda

data_repo_x_dda = data_all.loc[cond1 & cond2 & cond3].copy()

In [ ]:
data_repo_x_dda = data_repo_x_dda[data_repo_x_dda['date'] != dt.date(2025, 5, 5)]

In [ ]:
random_vta_promedio = np.random.uniform(0.3, 0.8, size=len(data_repo_x_dda))

In [ ]:
forecast_calcs = {
    "forecast_tricot":        data_repo_x_dda['vta_promedio'] * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
    "forecast_factor_hist":   data_repo_x_dda['vta_promedio'] * data_repo_x_dda['factor_historico'] * data_repo_x_dda['semana_vta'],
    "forecast_vta_promedio":  np.minimum(data_repo_x_dda['vta_promedio'], 1) * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
    "forecast_sv4":           data_repo_x_dda['vta_promedio'] * data_repo_x_dda['factor'] * 4,
    "forecast_f1":            data_repo_x_dda['vta_promedio'] * 1 * data_repo_x_dda['semana_vta'],
    "forecast_aleatorio":     random_vta_promedio * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
    "forecast_factor_hist_sv4":   data_repo_x_dda['vta_promedio'] * data_repo_x_dda['factor_historico'] * 4, 
    "forecast_ses":           data_repo_x_dda['SES_mean'] * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
    "forecast_holt":          data_repo_x_dda['Holt_mean'] * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
    "forecast_hw":            data_repo_x_dda['HW_mean'] * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
    "forecast_croston":       data_repo_x_dda['Croston_mean'] * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
    "forecast_tsb":           data_repo_x_dda['TSB_mean'] * data_repo_x_dda['factor'] * data_repo_x_dda['semana_vta'],
}

for col, expr in forecast_calcs.items():
    data_repo_x_dda[col] = expr.fillna(0).round(0).astype("UInt16")


# Valida

In [ ]:
forecast_dict = {
    "forecast_tricot": "real_sales",
    "forecast_factor_hist": "real_sales",
    "forecast_sv4": "real_sales_n4",
    "forecast_factor_hist_sv4": "real_sales_n4",
    "forecast_vta_promedio": "real_sales",
    "forecast_f1": "real_sales",
    "forecast_aleatorio": "real_sales",
    "forecast_ses": "real_sales",
    "forecast_holt": "real_sales",
    "forecast_hw": "real_sales",
    "forecast_croston": "real_sales",
    "forecast_tsb": "real_sales",
}

In [ ]:
dict_names = {
    "forecast_tricot": "Actual",
    "forecast_factor_hist": "Factor histórico",
    "forecast_sv4": "4 semanas de venta",
    "forecast_factor_hist_sv4": "4 semanas y factor histórico",
    "forecast_vta_promedio": "Venta prom. límitada a 1",
    "forecast_f1": "Factor fijo en 1",
    "forecast_aleatorio": "Vta. prom. aleatoria (entre 0.3 y 0.8)",
    "forecast_ses": "Suavizamiento exponencial",
    "forecast_holt": "Holt",
    "forecast_hw": "Holt-Winters",
    "forecast_croston": "Croston",
    "forecast_tsb": "TSB"
}

metrics = eval_forecast(data_repo_x_dda, forecast_dict)

metrics['forecast'] = metrics['forecast'].map(dict_names)

metrics

In [ ]:
colors = ['orange' if x == 'Actual' else 'gray' for x in metrics['forecast']]

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=metrics, x="wape", y="forecast", hue="forecast", palette=colors, legend=False)

# Línea vertical en el valor de Actual
actual_wape = metrics[metrics['forecast'] == 'Actual']['wape'].iloc[0]
plt.axvline(x=actual_wape, color='red', linestyle='--', alpha=0.7)

plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(0.25))
plt.gca().xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.title('Comparación de WAPE entre métodos de forecast')
plt.xlabel('WAPE')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=metrics, x="bias", y="forecast", hue="forecast", palette=colors, legend=False)

# Línea vertical en el valor de Actual
actual_bias = metrics[metrics['forecast'] == 'Actual']['bias'].iloc[0]
plt.axvline(x=actual_bias, color='red', linestyle='--', alpha=0.7)

plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(0.25))
plt.gca().xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.title('Comparación de BIAS entre métodos de forecast')
plt.xlabel('BIAS')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=metrics, x="rmse", y="forecast", hue="forecast", palette=colors, legend=False)

actual_mae = metrics[metrics['forecast'] == 'Actual']['rmse'].iloc[0]
plt.axvline(x=actual_mae, color='red', linestyle='--', alpha=0.7)

plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(5))

plt.title('Comparación de RMSE entre métodos de forecast')
plt.xlabel('RMSE')
plt.ylabel('')

plt.tight_layout()
plt.show()

# Ejemplo

In [ ]:
cod_producto = 680279
cod_talla = 105

data_sample = data_repo_x_dda[data_repo_x_dda['cod_producto'] == cod_producto].copy()
data_sample = data_sample[data_sample['cod_talla'] == cod_talla].copy()

In [ ]:
dict_names = {
    "forecast_tricot": "Actual",
    "forecast_factor_hist": "Factor histórico",
    "forecast_sv4": "4 semanas de venta",
    "forecast_vta_promedio": "Venta prom. límitada a 1",
    "forecast_f1": "Factor fijo en 1",
    "forecast_aleatorio": "Aleatorio"
}

In [ ]:
data_sample[['nombre_sucursal','cod_producto','cod_talla','week_number','date', 'can_final','real_sales', 'real_sales_n4',
             'forecast_tricot','forecast_factor_hist', 'forecast_sv4','forecast_vta_promedio','forecast_f1','forecast_aleatorio']].sort_values('can_final', ascending=False)